# Explainability and Clinical Alerts

In this notebook, we interpret the best performing model (e.g., LightGBM for the 24h prediction window) using SHAP (SHapley Additive exPlanations). 

Our explainability strategy aims to provide:
1. **Global insights:** Understanding which features drive the model's predictions overall (e.g., feature importance, SHAP summary plots).
2. **Local explanations:** Providing patient-specific insights to understand why a particular patient is flagged at high risk.
3. **Clinical Alerts:** Generating structured JSON alerts that highlight the top risk factors, which can be directly integrated into an EHR (Electronic Health Record) system to guide clinical decision-making.

In [ ]:
# Colab setup — mount Drive and cd into the project so ../data/... paths resolve.
# Safe anywhere: does nothing when not running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, sys
    PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
    os.chdir(os.path.join(PROJ, 'notebooks'))
    sys.path.insert(0, PROJ)
except ModuleNotFoundError:
    pass  # not on Colab — assume already running from notebooks/
import os
print('cwd =', os.getcwd())

In [1]:
import os
import sys
import json
import pandas as pd

sys.path.append('..')
from src import models, features
from src.explain import run_explainability

# Interpret the Stage-2 LightGBM model at the 24h horizon on the test split.
test_df = pd.read_parquet('../data/processed/test.parquet')
stage1, stage2 = models.load_models('../models')
s2_feats = features.get_all_features(test_df)

H = 24
model = stage2[H]['lgbm']['model']
X_test = test_df[s2_feats]
y_true = test_df[f'aki_label_{H}h'].values
y_prob = model.predict_proba(X_test)[:, 1]

# Conformal abstention threshold fitted in notebook 05 (travels in model_metadata.json).
conformal_qhat = None
try:
    with open('../models/model_metadata.json') as f:
        conformal_qhat = json.load(f).get('conformal_qhat_stage2_24h')
    print(f"Loaded conformal qhat = {conformal_qhat}")
except FileNotFoundError:
    print("model_metadata.json not found — alerts will run without abstention.")

output_dir = '../outputs/explain_stage2_24h'
os.makedirs(output_dir, exist_ok=True)
print(f"Explaining Stage-2 LGBM @ {H}h on {len(X_test)} test rows, {len(s2_feats)} features.")

Loaded conformal qhat = 0.6379081631454571
Explaining Stage-2 LGBM @ 24h on 1965 test rows, 198 features.


In [2]:
print("Running explainability pipeline (SHAP summary + waterfalls + clinical alerts)...")

summary = run_explainability(
    model=model,
    X_test=X_test,
    y_true=y_true,
    y_prob=y_prob,
    feature_names=s2_feats,
    output_dir=output_dir,
    conformal_qhat=conformal_qhat,
)

print("\nTop global features:", ", ".join(summary['top_features'][:5]))
print("Sample alerts generated:", summary['alert_count'])
if 'indeterminate_rate' in summary:
    print(f"Conformal abstention on test set: {summary['indeterminate_rate']:.1%} "
          f"(qhat={summary['conformal_qhat']:.3f})")
print("Artifacts saved to:", summary['output_dir'])

Running explainability pipeline (SHAP summary + waterfalls + clinical alerts)...


/Users/sengakabare/Desktop/Tunel/Projects/Personal/sentinel-poc/project_sentinel/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/Users/sengakabare/Desktop/Tunel/Projects/Personal/sentinel-poc/project_sentinel/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



Top global features: Age, cr_above_baseline, cr_ratio_baseline, urine_rate, BaseExcess
Sample alerts generated: 10
Conformal abstention on test set: 15.3% (qhat=0.638)
Artifacts saved to: /Users/sengakabare/Desktop/Tunel/Projects/Personal/sentinel-poc/project_sentinel/outputs/explain_stage2_24h


/Users/sengakabare/Desktop/Tunel/Projects/Personal/sentinel-poc/project_sentinel/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Sample Clinical Alert

To show what the EHR would display, we can print a sample alert from the JSON file generated above. The alert includes a risk score and a breakdown of the specific factors driving that risk, providing actionable context to the clinician.

In [3]:
# Load and print a sample alert to simulate the EHR display.
alerts_path = os.path.join(output_dir, 'sample_alerts.json')
try:
    with open(alerts_path, 'r') as f:
        alerts = json.load(f)

    if alerts:
        a = alerts[0]
        print("=" * 56)
        print("🚨 EHR CLINICAL ALERT — Sepsis-Associated AKI 🚨")
        print("=" * 56)
        print(f"Patient ID     : {a['patient_id']}")
        print(f"Risk level     : {a['risk_level']}  (score {a['risk_score']:.2f})")
        print(f"AKI prob (24h) : {a['aki_probability_24h']}")
        print("\nTop contributing factors:")
        for c in a['top_contributors']:
            print(f"  ⚠️  {c['feature']:32s} {c['direction']:8s} "
                  f"(value {c['value']}, SHAP {c['shap_impact']:+.3f})")
        print(f"\nRecommended action: {a['recommended_action']}")
        print("=" * 56)
    else:
        print("No alerts were generated.")

except FileNotFoundError:
    print("Alerts file not found. Ensure `run_explainability` ran successfully.")

🚨 EHR CLINICAL ALERT — Sepsis-Associated AKI 🚨
Patient ID     : 931
Risk level     : HIGH  (score 0.85)
AKI prob (24h) : 85%

Top contributing factors:
  ⚠️  Creatinine above baseline        RISING   (value 0.4, SHAP +1.394)
  ⚠️  Weight kg                        LOW      (value 58.0, SHAP -0.724)
  ⚠️  Platelets min 12h                HIGH     (value 48.0, SHAP +0.412)

Recommended action: URGENT: Consider immediate nephrology consultation, initiate fluid balance monitoring, and review nephrotoxic medications
